In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

C:\Users\MyPC\AppData\Local\Temp\ipykernel_12304\1403784872.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


#### As per RAG pipeline: Load Document -> Split/Chunk document -> Vector Embedding -> Storing Vectors in Vector DB
#### *************************************************************************
#### Step 1. We load the pdf document

In [15]:
path=r"D:\bloodtestreport.pdf"
loader=PyPDFLoader(path)
docs=loader.load()

#### Step 2. We split the documents data in chunks

In [5]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splitter_data=splitter.split_documents(docs)
len(splitter_data)

17

#### Step 3. We will do vector embedding of chunked data

In [6]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

#### Step 4. We will store the Embeddings in Vector DB (Chroma) and it will automaically embedd the embeddings object

In [7]:
vector_store=Chroma.from_documents(
    embedding=embeddings,
    documents=splitter_data,
    persist_directory="./chroma_db"
)

In [8]:
llm=ChatOpenAI(model="gpt-5")

#### Chain - Context Generator | Prompt | LLM | strParser

In [9]:
def get_context(query:str):
    data=vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context +=doc.page_content +"\n"

    return {
        "context":context,
        "question":query
    }
        

In [10]:
promptTemplate=PromptTemplate.from_template(
    """
    You are a helpful assitant and provide the answer based on the context for
    the user question. If you don't know the answer then you can say 'I don't know'.
    Context: {context}
    Question:{question}
    """
)

In [11]:
rag_chain=get_context| promptTemplate | llm

In [ ]:
res=rag_chain.invoke("give only hemoglobin data")
print(res.content)

नीचे आपकी उपलब्ध रिपोर्ट का आसान सारांश दिया है। जो मान रिपोर्ट के हिस्से में नहीं दिखे, उनके बारे में मैं पक्का नहीं कह सकता—कृपया वे मान साझा करें।

कौन‑कौन सी जांच हुई
- CBC (Complete Blood Count)
- LFT (Liver Function Test)
- Typhi Dot IgG और Typhi Dot IgM

रिपोर्ट में जो स्पष्ट दिख रहा है
- MCHC (Mean Corpuscular Hemoglobin Concentration): 33.9 g/dL (नॉर्मल 31.5–34.5) – सामान्य है।
  इसका मतलब: आपकी लाल रक्त कोशिकाओं में हीमोग्लोबिन की सांद्रता ठीक है।
- Typhi Dot IgG: Negative – पुराने/पूर्व टाइफॉयड संक्रमण की एंटीबॉडी नहीं मिलीं या अभी बनकर नहीं आईं। केवल IgG नेगेटिव होने से वर्तमान (एक्टिव) टाइफॉयड का पक्का निष्कर्ष नहीं निकलता।

रिपोर्ट में जो दिखाई नहीं दे रहा
- CBC के बाकी मान: हीमोग्लोबिन (Hb), WBC, प्लेटलेट्स आदि।
- LFT के मान: बिलीरुबिन, SGPT/ALT, SGOT/AST, एल्बुमिन आदि। (रिपोर्ट में केवल एल्बुमिन का सामान्य रेंज दिखा, आपका असली मान नहीं दिखा।)
- Typhi Dot IgM का परिणाम भी नजर नहीं आ रहा।

इन जांचों का मतलब (संक्षेप में)
- CBC: खून में हीमोग्लोबिन, इन्फेक्शन (WBC), और ब्ल